Step 1：时间筛选（快速）实现思路
遍历 Sentinel-1 元数据（按行）；

对于每一条 Sentinel-1 记录，从 CS2 文件中快速查找是否有轨迹点落在其 ±2 小时内；

若找到，就将该对（Sentinel-1 文件名，CS2 文件路径）加入候选。

In [ ]:
import geopandas as gpd
import xarray as xr
import numpy as np
from shapely.geometry import Point
import os

region = gpd.read_file(r'C:\Users\TJ002\Desktop\code\Cal_code_data\NWP_orbit_processing\Arctic_Canada_North.shp')
def clip_nc_by_region(nc_path, region_gdf):
    """
    读取 CryoSat-2 .nc 文件，只保留落在 region 范围内的点位
    """
    ds = xr.open_dataset(nc_path)
    lats = ds['lat_01'].values
    lons = ds['lon_01'].values

    # 构建点的 GeoSeries
    points = gpd.GeoSeries([Point(lon, lat) for lon, lat in zip(lons, lats)], crs="EPSG:4326")

    # region_gdf 已经是 GeoDataFrame，可能是多多边形，做 union
    region_union = region_gdf.unary_union

    # 使用 contains 向量判断哪些点在区域内
    mask = points.within(region_union)
    selected_lats = lats[mask]
    selected_lons = lons[mask]
    selected_times = ds['time_cor_01'].values[mask]

    # 若无点落在区域中，返回 None
    if len(selected_lats) == 0:
        return None

    return {
        "lat": selected_lats,
        "lon": selected_lons,
        "time": selected_times,
        "path": nc_path
    }


In [ ]:
import pandas as pd
import geopandas as gpd
import xarray as xr
import numpy as np
import os
from shapely.geometry import Point
from datetime import timedelta
from tqdm import tqdm

def clip_nc_by_region(nc_path, region_gdf):
    """
    读取 CryoSat-2 .nc 文件，只保留落在 region 范围内的点位
    """
    ds = xr.open_dataset(nc_path)
    lats = ds['lat_01'].values
    lons = ds['lon_01'].values

    points = gpd.GeoSeries([Point(lon, lat) for lon, lat in zip(lons, lats)], crs="EPSG:4326")
    region_union = region_gdf.unary_union

    mask = points.within(region_union)
    if not mask.any():
        return None

    return {
        "lat": lats[mask],
        "lon": lons[mask],
        "time": ds['time_cor_01'].values[mask],
        "path": nc_path
    }

# -------------------------
# 加载 region shapefile
# -------------------------
region = gpd.read_file(r'C:\Users\TJ002\Desktop\code\Cal_code_data\NWP_orbit_processing\Arctic_Canada_North.shp')

# -------------------------
# 加载 Sentinel-1 Metadata
# -------------------------
s1_df = pd.read_csv(r"E:\NWP\Sentinel1\Metadata\2023\sentinel1_2023_all.csv", parse_dates=["Start Time", "End Time"])
# 移除时区信息，避免与 CryoSat-2 时间冲突
s1_df["Start Time"] = s1_df["Start Time"].dt.tz_localize(None)
s1_df["End Time"] = s1_df["End Time"].dt.tz_localize(None)

# -------------------------
# Clip CryoSat-2 by region first
# -------------------------
cs2_clipped_summary = []

year_path = r"E:\NWP\CS2\2023"  # Change to your CS2 root folder


for root, _, files in os.walk(year_path):
    for f in files:
        if f.endswith(".nc"):
            full_path = os.path.join(root, f)
            try:
                clipped = clip_nc_by_region(full_path, region)
                if clipped is not None:
                    times = pd.to_datetime(clipped["time"])
                    cs2_clipped_summary.append({
                        "cs2_path": full_path,
                        "cs2_start": times[0],
                        "cs2_end": times[-1],
                        "lat": clipped["lat"],
                        "lon": clipped["lon"],
                        "time": clipped["time"]
                    })
            except Exception as e:
                print(f"❌ 读取失败 {full_path}: {e}")

print(f"✅ 区域内 CS2 文件数：{len(cs2_clipped_summary)}")
# -------------------------
# 进行时间匹配
# -------------------------
time_matches = []
for _, s1_row in tqdm(s1_df.iterrows(), total=len(s1_df)):
    s1_start, s1_end = s1_row["Start Time"], s1_row["End Time"]
    s1_lon, s1_lat = s1_row["Center Lon"], s1_row["Center Lat"]

    for cs2 in cs2_clipped_summary:
        # 判断是否时间重叠（±2小时窗口）
        if (cs2["cs2_end"] >= s1_start - timedelta(hours=2)) and (cs2["cs2_start"] <= s1_end + timedelta(hours=2)):
            # 简易空间粗筛（S1 中心点在 CS2 点 bbox 内）
            if (cs2["lat"].min() <= s1_lat <= cs2["lat"].max()) and (cs2["lon"].min() <= s1_lon <= cs2["lon"].max()):
                time_matches.append({
                    "sceneName": s1_row["Granule Name"],
                    "s1_start": s1_start,
                    "s1_end": s1_end,
                    "s1_lon": s1_lon,
                    "s1_lat": s1_lat,
                    "cs2_path": cs2["cs2_path"],
                    "cs2_start": cs2["cs2_start"],
                    "cs2_end": cs2["cs2_end"]
                })

# -------------------------
# 保存匹配结果
# -------------------------
matches_df = pd.DataFrame(time_matches)
# os.makedirs(r"E:\NWP\CS2_S1_matched", exist_ok=True)
# matches_df.to_csv(r"E:\NWP\CS2_S1_matched\region_time_matched.csv", index=False)
# print(f"时间+区域+空间粗筛）：{len(matches_df)}")


✅ 区域内 CS2 文件数：2641


100%|██████████| 15904/15904 [01:34<00:00, 168.26it/s]


✅ 最终匹配数（时间+区域+空间粗筛）：870


 步骤 1：构建 Sentinel-1 footprint polygon

In [18]:
from shapely.geometry import Polygon, Point
import geopandas as gpd

def build_s1_polygon(row):
    coords = [
        (row['Near Start Lon'], row['Near Start Lat']),
        (row['Far Start Lon'], row['Far Start Lat']),
        (row['Far End Lon'], row['Far End Lat']),
        (row['Near End Lon'], row['Near End Lat']),
        (row['Near Start Lon'], row['Near Start Lat'])  # 闭合
    ]
    return Polygon(coords)


遍历匹配项，查找穿越的 CS2 点

In [19]:
import xarray as xr
from datetime import timedelta

results = []

for _, match in tqdm(matches_df.iterrows(), total=len(matches_df)):
    # 找到对应的 Sentinel-1 行信息
    s1_row = s1_df[s1_df["Granule Name"] == match["sceneName"]].iloc[0]
    s1_poly = build_s1_polygon(s1_row)
    s1_start, s1_end = match["s1_start"], match["s1_end"]
    time_window_start = s1_start - timedelta(hours=2)
    time_window_end = s1_end + timedelta(hours=2)

    # 打开对应 CS2 文件
    ds = xr.open_dataset(match["cs2_path"])
    lat = ds["lat_01"].values
    lon = ds["lon_01"].values
    time = pd.to_datetime(ds["time_cor_01"].values)

    # 遍历每个点，判断是否同时满足：
    # 1. 落入 Sentinel-1 polygon 内；
    # 2. time 在 ±2h 内
    for lo, la, t in zip(lon, lat, time):
        if time_window_start <= t <= time_window_end:
            pt = Point(lo, la)
            if s1_poly.contains(pt):
                results.append({
                    "sceneName": match["sceneName"],
                    "cs2_path": match["cs2_path"],
                    "cs2_time": t,
                    "cs2_lat": la,
                    "cs2_lon": lo
                })


100%|██████████| 870/870 [00:16<00:00, 53.55it/s]


In [20]:
cs2_cross_points_df = pd.DataFrame(results)
# cs2_cross_points_df.to_csv("cs2_points_within_sentinel1_footprint.csv", index=False)
print(f"✅ 最终匹配数（时间+区域+空间粗筛）：{len(cs2_cross_points_df)}")
cs2_cross_points_df = pd.DataFrame(results)
# cs2_cross_points_df.to_csv("cs2_points_within_sentinel1_footprint.csv", index=False)
# print(f"✅ 最终匹配数（时间+区域+空间粗筛）：{len(cs2_cross_points_df)}")

✅ 最终匹配数（时间+区域+空间粗筛）：35038


In [22]:
# 假设你已有以下 DataFrame，来自之前筛选：
# columns: sceneName, cs2_path, cs2_time, cs2_lat, cs2_lon

# 1. 合并 Sentinel-1 metadata 中的 URL、时间信息
s1_info = s1_df[["Granule Name", "Start Time", "End Time", "URL"]].copy()
s1_info.columns = ["sceneName", "s1_start", "s1_end", "s1_url"]

merged = pd.merge(cs2_cross_points_df, s1_info, on="sceneName", how="left")

# 2. 按 sceneName + cs2_path 分组，统计穿越点数量和时间范围
grouped = merged.groupby(["sceneName", "cs2_path"]).agg({
    "s1_url": "first",
    "s1_start": "first",
    "s1_end": "first",
    "cs2_time": ["min", "max", "count"]
}).reset_index()

# 3. 展平多级列名
grouped.columns = [
    "sceneName", "cs2_nc", "s1_url", "s1_start", "s1_end",
    "cs2_time_min", "cs2_time_max", "num_matched_points"
]

# 4. 保存为 CSV
grouped.to_csv("valid_cs2_s1_pairs.csv", index=False)
print(grouped)

                                             sceneName  \
0    S1A_EW_GRDM_1SDH_20230413T143709_20230413T1438...   
1    S1A_EW_GRDM_1SDH_20230427T142145_20230427T1422...   
2    S1A_EW_GRDM_1SDH_20230430T130640_20230430T1307...   
3    S1A_EW_GRDM_1SDH_20230501T134746_20230501T1348...   
4    S1A_EW_GRDM_1SDH_20230502T125011_20230502T1251...   
..                                                 ...   
396  S1A_IW_SLC__1SDH_20230703T141341_20230703T1414...   
397  S1A_IW_SLC__1SDH_20230704T131558_20230704T1316...   
398  S1A_IW_SLC__1SDH_20230704T131624_20230704T1316...   
399  S1A_IW_SLC__1SDH_20230704T145440_20230704T1455...   
400  S1A_IW_SLC__1SDH_20230825T205114_20230825T2051...   

                                                cs2_nc  \
0    E:\NWP\CS2\2023\CS_OFFL_SIR_SIN_2__20230413T16...   
1    E:\NWP\CS2\2023\CS_OFFL_SIR_SIN_2__20230427T16...   
2    E:\NWP\CS2\2023\CS_OFFL_SIR_SIN_2__20230430T13...   
3    E:\NWP\CS2\2023\CS_OFFL_SIR_SIN_2__20230501T14...   
4    E:\NWP\C

In [17]:
time_df = pd.DataFrame(time_matches)
# 4. 精细空间匹配（使用 bounding box 粗筛）
final_matches = []

for _, row in tqdm(time_df.iterrows(), total=len(time_df)):
    try:
        ds = xr.open_dataset(row["cs2_path"])
        lats = ds["lat_01"].values
        lons = ds["lon_01"].values

        min_lat, max_lat = lats.min(), lats.max()
        min_lon, max_lon = lons.min(), lons.max()

        # 简单 bounding box 判断
        lat, lon = row["s1_lat"], row["s1_lon"]
        if (min_lat <= lat <= max_lat) and (min_lon <= lon <= max_lon):
            final_matches.append(row)

    except Exception as e:
        print(f"空间匹配失败: {row['cs2_path']} — {e}")

final_df = pd.DataFrame(final_matches)

# 5. 保存最终匹配结果
out_path = r"E:\NWP\CS2_S1_matched\time_space_matched_pairs_2023.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
final_df.to_csv(out_path, index=False)
print(f"✅ 最终匹配成功数：{len(final_df)}")

100%|██████████| 870/870 [00:12<00:00, 72.23it/s]


✅ 最终匹配成功数：870


In [ ]:


cs2_filtered_results = []

for root, _, files in os.walk(r"E:\NWP\CS2\test2023"):
    for f in files:
        if f.endswith(".nc"):
            nc_path = os.path.join(root, f)
            clipped = clip_nc_by_region(nc_path, region)
            if clipped is not None:
                cs2_filtered_results.append(clipped)


In [6]:
import pandas as pd
import geopandas as gpd
import xarray as xr
import numpy as np
import os
from shapely.geometry import Point
from datetime import timedelta
from tqdm import tqdm

def clip_nc_by_region(nc_path, region_gdf):
    """
    读取 CryoSat-2 .nc 文件，只保留落在 region 范围内的点位
    """
    ds = xr.open_dataset(nc_path)
    lats = ds['lat_01'].values
    lons = ds['lon_01'].values

    points = gpd.GeoSeries([Point(lon, lat) for lon, lat in zip(lons, lats)], crs="EPSG:4326")
    region_union = region_gdf.unary_union

    mask = points.within(region_union)
    if not mask.any():
        return None

    return {
        "lat": lats[mask],
        "lon": lons[mask],
        "time": ds['time_cor_01'].values[mask],
        "path": nc_path
    }

# -------------------------
# 加载 region shapefile
# -------------------------
region = gpd.read_file(r'C:\Users\TJ002\Desktop\code\Cal_code_data\NWP_orbit_processing\Arctic_Canada_North.shp')

# -------------------------
# 加载 Sentinel-1 Metadata
# -------------------------
s1_df = pd.read_csv(r"E:\NWP\Sentinel1\Metadata\2023\sentinel1_2023_all.csv", parse_dates=["Start Time", "End Time"])

# -------------------------
# Clip CryoSat-2 by region first
# -------------------------
cs2_clipped_summary = []

base_dir = r"E:\NWP\CS2"
for year_folder in os.listdir(base_dir):
    year_path = os.path.join(base_dir, year_folder)
    if not os.path.isdir(year_path):
        continue

    for root, _, files in os.walk(year_path):
        for f in files:
            if f.endswith(".nc"):
                full_path = os.path.join(root, f)
                try:
                    clipped = clip_nc_by_region(full_path, region)
                    if clipped is not None:
                        times = pd.to_datetime(clipped["time"])
                        cs2_clipped_summary.append({
                            "cs2_path": full_path,
                            "cs2_start": times[0],
                            "cs2_end": times[-1],
                            "lat": clipped["lat"],
                            "lon": clipped["lon"],
                            "time": clipped["time"]
                        })
                except Exception as e:
                    print(f"❌ 读取失败 {full_path}: {e}")

print(f"✅ 区域内 CS2 文件数：{len(cs2_clipped_summary)}")


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import xarray as xr
import os
from datetime import timedelta

# 1. 加载 Sentinel-1 metadata
s1_df = pd.read_csv(r"E:\NWP\Sentinel1\Metadata\2023\sentinel1_2023_all.csv", parse_dates=["Start Time", "End Time"])

# 2. 遍历 CryoSat-2 年份文件夹下所有 .nc 文件
cryo_nc_files = []
for root, _, files in os.walk(r"E:\NWP\CS2\2023"):
    for f in files:
        if f.endswith(".nc"):
            cryo_nc_files.append(os.path.join(root, f))

# 3. 构建匹配结果列表
time_matches = []

for nc_path in cryo_nc_files:
    ds = xr.open_dataset(nc_path)
    times = pd.to_datetime(ds["time_cor_01"].values)

    # 快速筛选所有S1条目中，和该CS2轨迹时间段重叠的（±2小时）
    s1_mask = (s1_df["End Time"] >= times[0] - timedelta(hours=2)) & \
              (s1_df["Start Time"] <= times[-1] + timedelta(hours=2))
    if s1_mask.any():
        for _, row in s1_df[s1_mask].iterrows():
            s1_start, s1_end = row["Start Time"], row["End Time"]
            in_window = (times >= s1_start - timedelta(hours=2)) & (times <= s1_end + timedelta(hours=2))
            if in_window.any():
                time_matches.append({
                    "sceneName": row["sceneName"],
                    "s1_start": s1_start,
                    "s1_end": s1_end,
                    "cs2_nc_path": nc_path
                })

# 保存匹配结果
matches_df = pd.DataFrame(time_matches)
matches_df.to_csv("time_matched_pairs.csv", index=False)
